# Background purity

The background class must be empty of catalogued galaxies and share
everything else — provenance, conditions, masking — with the galaxy class.
This notebook puts eyes on a few hundred accepted and rejected candidates
and quantifies the survival fraction the driver reports. Run
`scripts/fetch_backgrounds.py` first.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))

root = REPO / "data" / "sga"
galaxies = root / "samples" / "galaxies"
backgrounds = root / "samples" / "backgrounds"
print("data root:", root)

In [ ]:
def show(ax, image, sigma=None):
    """Asinh stretch scaled to the frame's own noise; grayscale, origin low."""
    if sigma is None:
        sigma = 1.4826 * np.median(np.abs(image - np.median(image))) or 1e-3
    ax.imshow(np.arcsinh(image / (3.0 * sigma)), origin="lower",
              cmap="gray", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])


def load_manifest(path):
    import csv
    with open(path) as f:
        return [row for row in csv.DictReader(f)]


candidates = load_manifest(backgrounds / "candidates.csv")
accepted = [c for c in candidates if c["status"] == "accepted"]
rejected = [c for c in candidates if c["status"] == "rejected"]
print(f"{len(candidates)} candidates: {len(accepted)} accepted "
      f"({len(accepted) / len(candidates):.1%}), {len(rejected)} rejected")

## Why candidates die

Every rejection is structural — footprint, an SGA galaxy's extent
(catalogue ellipse or DR9's own GALAXY maskbit), or coverage. Nothing is
rejected for faint sources or bright stars; those are masked and kept.

In [ ]:
from collections import Counter

reasons = Counter(c["reason"] for c in rejected)
order = ["sga_overlap", "sga_maskbit", "footprint", "coverage", "fetch"]
counts = [reasons.get(r, 0) for r in order]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(order[::-1], counts[::-1], color="C0")
for y, value in enumerate(counts[::-1]):
    ax.text(value, y, f" {value}", va="center")
ax.set_xlabel("rejected candidates")
ax.set_title(f"Rejections ({len(rejected)} of {len(candidates)})")
fig.tight_layout()

In [ ]:
from collections import defaultdict

per_brick = defaultdict(lambda: [0, 0])
for c in candidates:
    per_brick[c["brick"]][1] += 1
    if c["status"] == "accepted":
        per_brick[c["brick"]][0] += 1
fractions = np.array([a / t for a, t in per_brick.values()])
in_parent = np.mean([int(c["in_parent_brick"]) for c in accepted])
seps = np.array([float(c["sep_arcsec"]) for c in accepted])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fractions, bins=20, color="C0")
axes[0].set_xlabel("survival fraction"); axes[0].set_ylabel("bricks")
axes[0].set_title(f"Per-brick survival (median {np.median(fractions):.0%})")
axes[1].hist(seps, bins=40, color="C0")
axes[1].axvline(np.median(seps), color="C3",
                label=f"median {np.median(seps):.0f}\"")
axes[1].set_xlabel("separation from parent (arcsec)"); axes[1].legend()
axes[1].set_title(f"Separations ({in_parent:.0%} share the parent brick)")
fig.tight_layout()

## Accepted stamps, by eye

r band, asinh-stretched to each frame's own noise. These should read as
empty sky: faint small sources are expected (and deliberately kept), any
resolved galaxy filling a frame is a failure of the clearance tests.

In [ ]:
manifest = {r["file"]: r for r in load_manifest(backgrounds / "manifest.csv")
            if r["status"] == "written"}
files = sorted(manifest)
rng = np.random.default_rng(1)
picks = rng.choice(len(files), min(100, len(files)), replace=False)
fig, axes = plt.subplots(10, 10, figsize=(14, 14))
for ax, i in zip(axes.ravel(), picks):
    with fits.open(backgrounds / files[i]) as sample:
        show(ax, sample["SCI"].data[1])
for ax in axes.ravel()[len(picks):]:
    ax.axis("off")
fig.suptitle("accepted backgrounds (r)", y=0.995)
fig.tight_layout()

## Rejected candidates, by eye

Rejected positions are never written to disk, but most fall in bricks the
drivers already mirrored, so they can be cut on the fly. Each stamp is
labelled with its rejection reason — the SGA-overlap ones should visibly
contain (or graze) a catalogued galaxy.

In [ ]:
from src.cutout import Coadd, brick_dir, cut

shown = 0
fig, axes = plt.subplots(6, 8, figsize=(14, 11))
axes_flat = iter(axes.ravel())
for c in rejected:
    if c["reason"] in ("footprint", "fetch"):
        continue
    directory = brick_dir(root, c["brick"])
    if not (directory / f"legacysurvey-{c['brick']}-image-r.fits.fz").exists():
        continue
    try:
        ax = next(axes_flat)
    except StopIteration:
        break
    brick = Coadd(brick_dir(root, c["brick"]), c["brick"])
    sample = cut(brick, float(c["ra"]), float(c["dec"]), int(c["size_px"]))
    show(ax, sample["sci"][1])
    ax.set_title(c["reason"], fontsize=8)
    shown += 1
for ax in axes_flat:
    ax.axis("off")
fig.suptitle(f"rejected candidates ({shown} shown)", y=0.995)
fig.tight_layout()

## Class symmetry

The two classes must not separate on masking statistics. Bright-object and
source-footprint fractions are drawn per frame from the manifests of both
drivers; backgrounds carrying *fewer* faint sources than galaxy
neighbourhoods would hand the discriminator a shortcut, so similar
distributions are the goal, not maximal emptiness.

In [ ]:
galaxy_manifest = [r for r in load_manifest(galaxies / "manifest.csv")
                   if r["status"] == "written"]
background_manifest = list(manifest.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key in zip(axes, ("bright_frac", "source_frac")):
    bins = np.linspace(0, 1, 40)
    for rows, color, label in ((galaxy_manifest, "C0", "galaxies"),
                               (background_manifest, "C1", "backgrounds")):
        values = np.array([float(r[key]) for r in rows])
        ax.hist(values, bins=bins, histtype="step", density=True,
                color=color, label=label)
    ax.set_xlabel(key); ax.set_yscale("log"); ax.legend()
axes[0].set_title("bright-object mask fraction")
axes[1].set_title("source mask fraction")
fig.tight_layout()

galaxy_bit = np.array([float(r["galaxy_bit_frac"]) for r in background_manifest])
print(f"backgrounds with any DR9 GALAXY-bit pixel: {(galaxy_bit > 0).mean():.1%} "
      "(0% expected — the prescreen rejects them)")